# Validation-only FSD ablations
Teacher-depth and anchor-weight sweeps using the immutable training-derived validation manifest. The official test set is never evaluated.

In [ ]:
!pip install -q lpips scikit-image
from pathlib import Path
import hashlib, shutil, subprocess, sys, zipfile
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/FSD_Validation_Ablations')

def source_roots(base):
    return [p.parent for p in base.rglob('run_validation_ablations.py')
            if (p.parent/'src/train_distill.py').is_file()]

roots = source_roots(INPUT)
source_description = None
if not roots:
    extract_root = Path('/kaggle/working/uploaded_validation_ablation_source')
    if extract_root.exists(): shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)
    matched_archives = []
    for archive in INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(archive) as zf:
                names = zf.namelist()
                if (any(n.endswith('run_validation_ablations.py') for n in names) and
                    any(n.endswith('src/train_distill.py') for n in names)):
                    zf.extractall(extract_root)
                    matched_archives.append(archive)
        except zipfile.BadZipFile:
            pass
    roots = source_roots(extract_root)
    source_description = matched_archives
else:
    source_description = roots
if len(roots) != 1:
    visible = [str(p) for p in INPUT.iterdir()]
    raise RuntimeError(f'Expected one source root, found {roots}. Attached inputs: {visible}')
SOURCE_ROOT = roots[0]
if WORK.exists(): shutil.rmtree(WORK)
shutil.copytree(SOURCE_ROOT, WORK)

dataset_candidates = list(INPUT.rglob('LOL-v2/Real_captured/Train/Low'))
if not dataset_candidates: dataset_candidates = list(INPUT.rglob('Real_captured/Train/Low'))
if len(dataset_candidates) != 1:
    raise RuntimeError(f'Expected one LOL-v2 dataset, found {dataset_candidates}')
DATASET_ROOT = dataset_candidates[0].parents[2]

teachers = list(INPUT.rglob('teacher_illum_ft.pth'))
EXPECTED = 'a92c2988ced5cb0fb8e403fb143913d8fc2837fdb31dbeb1634aa3ca031697a2'
teachers = [p for p in teachers if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED]
if len(teachers) != 1:
    raise RuntimeError(f'Exact transferred teacher not found: {teachers}')
TEACHER = teachers[0]
print('Source:', source_description)
print('Dataset:', DATASET_ROOT)
print('Teacher:', TEACHER)


In [ ]:
# Seven 20-epoch validation-only runs. This is intentionally long.
cmd = [sys.executable, str(WORK/'run_validation_ablations.py'),
       '--dataset-root', str(DATASET_ROOT), '--teacher', str(TEACHER),
       '--epochs', '20', '--seed', '3407', '--eval-seed', '99173']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import pandas as pd
display(pd.read_csv(WORK/'results/validation_ablation_summary.csv'))
EVIDENCE = Path('/kaggle/working/fsd_validation_ablation_evidence')
if EVIDENCE.exists(): shutil.rmtree(EVIDENCE)
EVIDENCE.mkdir()
for path in (WORK/'results').rglob('*'):
    if path.is_file() and path.suffix.lower() in {'.csv', '.json'}:
        target = EVIDENCE/path.relative_to(WORK/'results')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
archive = shutil.make_archive('/kaggle/working/fsd_validation_ablation_evidence', 'zip', EVIDENCE)
print('Download:', archive)
print('Use Save Version before ending the session so checkpoints remain available.')
